# 03 — Statistical Testing

Test associations between predictors and the 30-day readmission target.

- **Categorical predictors**: chi-square test of independence.
- **Numeric predictors**: Welch's t-test (or Mann–Whitney U if the distribution is heavy-tailed).

Apply a multiple-testing correction (Benjamini–Hochberg) before interpreting p-values.

**Inputs**: `data/processed/cleaned.csv`


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

from src.config import PROCESSED_DIR, TARGET

In [3]:
# load the cleaned data from the previous notebook
df = pd.read_csv(PROCESSED_DIR / 'cleaned.csv')
df.shape

(3000, 15)

In [4]:
# separate the target, then split the features into numeric vs categorical
y = df[TARGET]
features = df.drop(columns=[TARGET])
numeric_cols = features.select_dtypes('number').columns.tolist()
categorical_cols = features.select_dtypes(exclude='number').columns.tolist()
print('numeric:', numeric_cols)
print('categorical:', categorical_cols)

numeric: ['age', 'bmi', 'bnp', 'sodium', 'creatinine', 'systolic_bp', 'heart_rate', 'ace_inhibitor', 'beta_blocker', 'diuretic', 'adherence_score', 'distance_to_hospital_km']
categorical: ['gender', 'income_level']


In [5]:
# chi-square test for each categorical feature: is it associated with readmission?
rows = []
for col in categorical_cols:
    contingency = pd.crosstab(df[col], y)
    chi2, p, dof, _ = stats.chi2_contingency(contingency)
    rows.append({'feature': col, 'test': 'chi2', 'stat': chi2, 'dof': dof, 'p_raw': p})
chi_results = pd.DataFrame(rows)
chi_results

,feature,test,stat,dof,p_raw
0,gender,chi2,0.000241,1,0.987606
1,income_level,chi2,0.450853,2,0.798176


In [6]:
# Welch's t-test for each numeric feature: do readmitted and non-readmitted groups differ?
rows = []
for col in numeric_cols:
    a = df.loc[y == 0, col].dropna()
    b = df.loc[y == 1, col].dropna()
    if len(a) < 2 or len(b) < 2:
        continue
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({
        'feature': col, 'test': 'welch_t',
        'stat': t, 'p_raw': p,
        'mean_neg': a.mean(), 'mean_pos': b.mean(),
    })
num_results = pd.DataFrame(rows)
num_results

,feature,test,stat,p_raw,mean_neg,mean_pos
0,age,welch_t,-5.023715,5.406868e-07,64.121744,66.875203
1,bmi,welch_t,-1.467541,1.423514e-01,28.003329,28.276358
2,bnp,welch_t,-8.859777,1.469716e-18,374.988109,451.130470
3,sodium,welch_t,2.735444,6.273938e-03,138.262412,137.843095
4,creatinine,welch_t,-2.455135,1.415006e-02,1.185283,1.229908
5,systolic_bp,welch_t,-1.939316,5.256546e-02,128.994904,130.438412
6,heart_rate,welch_t,0.670192,5.027937e-01,79.557507,79.179238
7,ace_inhibitor,welch_t,5.121346,3.251863e-07,0.560589,0.465964
8,beta_blocker,welch_t,4.700301,2.730347e-06,0.539071,0.452188
9,diuretic,welch_t,-0.858773,3.905434e-01,0.482446,0.498379


In [7]:
# we ran many tests, so correct the p-values (Benjamini-Hochberg) before trusting them
all_results = pd.concat([chi_results, num_results], ignore_index=True)
reject, p_adj, _, _ = multipletests(all_results['p_raw'], method='fdr_bh')
all_results['p_fdr'] = p_adj
all_results['significant_fdr_0_05'] = reject
all_results.sort_values('p_fdr')

,feature,test,stat,dof,p_raw,mean_neg,mean_pos,p_fdr,significant_fdr_0_05
12,adherence_score,welch_t,10.683264,NaN,4.334020e-26,0.727384,0.659295,6.067628e-25,True
4,bnp,welch_t,-8.859777,NaN,1.469716e-18,374.988109,451.130470,1.028801e-17,True
9,ace_inhibitor,welch_t,5.121346,NaN,3.251863e-07,0.560589,0.465964,1.517536e-06,True
2,age,welch_t,-5.023715,NaN,5.406868e-07,64.121744,66.875203,1.892404e-06,True
10,beta_blocker,welch_t,4.700301,NaN,2.730347e-06,0.539071,0.452188,7.644971e-06,True
13,distance_to_hospital_km,welch_t,-3.698488,NaN,2.212040e-04,24.607135,26.533549,5.161427e-04,True
5,sodium,welch_t,2.735444,NaN,6.273938e-03,138.262412,137.843095,1.254788e-02,True
6,creatinine,welch_t,-2.455135,NaN,1.415006e-02,1.185283,1.229908,2.476261e-02,True
7,systolic_bp,welch_t,-1.939316,NaN,5.256546e-02,128.994904,130.438412,8.176850e-02,False
3,bmi,welch_t,-1.467541,NaN,1.423514e-01,28.003329,28.276358,1.992920e-01,False


## Notes

- Welch's t-test is sensible for unequal variances. For heavily skewed numeric features (BNP, creatinine) consider switching to `mannwhitneyu`.
- Report effect sizes alongside p-values (Cohen's d for numerics, Cramér's V for chi-square) — the proposal expects practical, not just statistical, significance.
- Save the result table to `reports/` for the writeup.
